# MScFE 610 Financial Econometrics — Group Work Project 1
### Group 17817 · Module 6 · Technical notebook

This notebook holds the code and output for every computational answer in the report. Each section
is headed by the question it answers. Group 17817 is an **odd** group, so Problem 3 uses
`FE-GWP1_model_selection_1.csv`. Figures are written to `figures/`, data are read from `data/`.

| Section | Question | Content |
|---|---|---|
| Setup | — | Libraries, master seed, plotting conventions |
| 1 | **1e** | Omitted-variable bias by Monte Carlo simulation |
| 2 | **2b, 2c** | Sensitivity of OLS to outliers; the high-leverage limit |
| 3 | **3a, 3b** | Model selection on dataset 1; consistency of BIC versus AIC |
| 4 | **4c–4h** | Interpretation of the log-wage equation |
| 5 | **5c** | Unit roots, Johansen cointegration tests, VECM |
| 6 | **6a, 6b** | AR(1) MA(∞) representation; AR(2) autocorrelations |

Questions that are purely analytical (1a–1d, 2a, 4a–4b, 5a–5b) are answered in the written report.

## Setup

In [ ]:
import itertools, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss, acf
from statsmodels.tsa.vector_ar.vecm import coint_johansen, VECM, select_coint_rank, select_order
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_ljungbox, linear_reset
from statsmodels.stats.outliers_influence import OLSInfluence, variance_inflation_factor

warnings.filterwarnings("ignore")
pd.set_option("display.width", 130, "display.float_format", lambda v: f"{v:9.4f}")
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False, "figure.autolayout": True})
BLUE, RED, GREEN, PURPLE, GREY = "#2A6FDB", "#D1495B", "#3C8D5A", "#8C6BB1", "#6B7280"

SEED = 17817          # the group number, used as the master seed so every result is reproducible
DATA = "data"         # input CSV files
FIG  = "figures"      # every chart is saved here for the report
os.makedirs(FIG, exist_ok=True)

def save_fig(name):
    """Save the current figure to figures/<name>.png at print resolution, then display it."""
    plt.savefig(f"{FIG}/{name}.png", dpi=200, bbox_inches="tight")
    plt.show()

print("Master random seed:", SEED)

## Question 1e — Omitted-variable bias in a controlled simulation
### Omitted-variable bias in a controlled simulation

Data-generating process, with every parameter known by construction:

$$Y_i = a + bX_i + cZ_i + e_i,\qquad a=1,\; b=2,\; c=1.5,$$

where $(X_i, Z_i)$ are drawn from a bivariate normal with unit variances and correlation
$\rho=0.7$, and $e_i \sim N(0,1)$ is drawn independently of both regressors. We then estimate the
model twice: once correctly, and once with $Z$ omitted.

Theory predicts that omitting $Z$ drives the estimator of $b$ to
$\; \operatorname{plim}\hat b_{\text{short}} = b + c\,\dfrac{\operatorname{Cov}(X,Z)}{\operatorname{Var}(X)} = 2 + 1.5(0.7) = 3.05.$

In [ ]:
A_TRUE, B_TRUE, C_TRUE = 1.0, 2.0, 1.5
RHO, SX, SZ, SE = 0.7, 1.0, 1.0, 1.0

def draw(n, rng, rho=RHO):
    '''One sample from the known data-generating process.'''
    cov = [[SX**2, rho*SX*SZ], [rho*SX*SZ, SZ**2]]
    XZ = rng.multivariate_normal([0, 0], cov, size=n)
    X, Z = XZ[:, 0], XZ[:, 1]
    Y = A_TRUE + B_TRUE*X + C_TRUE*Z + rng.normal(0, SE, n)
    return X, Z, Y

theory_bias = C_TRUE * RHO * SZ / SX
print(f"True b = {B_TRUE};  theoretical asymptotic value of the misspecified estimator "
      f"= b + c*rho = {B_TRUE + theory_bias:.4f}")

In [ ]:
# --- 1e(i) one sample of 200 observations, estimated both ways -------------------------------
rng = np.random.default_rng(SEED)
X, Z, Y = draw(200, rng)
full  = sm.OLS(Y, sm.add_constant(np.column_stack([X, Z]))).fit()
short = sm.OLS(Y, sm.add_constant(X)).fit()
aux   = sm.OLS(Z, sm.add_constant(X)).fit()          # auxiliary regression of the omitted variable on X

print("Correct specification   Y ~ X + Z")
print(f"   a = {full.params[0]:.4f} (se {full.bse[0]:.4f})   b = {full.params[1]:.4f} (se {full.bse[1]:.4f})"
      f"   c = {full.params[2]:.4f} (se {full.bse[2]:.4f})   R2 = {full.rsquared:.4f}   s = {np.sqrt(full.mse_resid):.4f}")
print("Misspecified            Y ~ X          (Z omitted)")
print(f"   a = {short.params[0]:.4f} (se {short.bse[0]:.4f})   b = {short.params[1]:.4f} (se {short.bse[1]:.4f})"
      f"                        R2 = {short.rsquared:.4f}   s = {np.sqrt(short.mse_resid):.4f}")
print(f"\nAuxiliary regression Z on X: slope = {aux.params[1]:.4f}")
print(f"Algebraic identity  b_short = b_full + c_full * slope(Z on X):"
      f"  {short.params[1]:.6f}  =  {full.params[1] + full.params[2]*aux.params[1]:.6f}")
print(f"\nCorrelation of the short-model residual with X = {np.corrcoef(short.resid, X)[0,1]:.2e}  (zero by construction)")
print(f"Correlation of the short-model residual with Z = {np.corrcoef(short.resid, Z)[0,1]:.4f}  "
      "(the omitted variable now lives inside the disturbance)")

In [ ]:
# --- 1e(ii) does a larger sample fix the problem?  5,000 replications at each sample size -----
NS, REP = [25, 50, 100, 200, 500, 1000, 5000, 20000], 5000
rows = []
for n in NS:
    rng = np.random.default_rng(SEED + n)
    bf, bs, sef, ses = [], [], [], []
    for _ in range(REP):
        X, Z, Y = draw(n, rng)
        f = sm.OLS(Y, sm.add_constant(np.column_stack([X, Z]))).fit()
        s = sm.OLS(Y, sm.add_constant(X)).fit()
        bf.append(f.params[1]); bs.append(s.params[1]); sef.append(f.bse[1]); ses.append(s.bse[1])
    bf, bs, sef, ses = map(np.array, (bf, bs, sef, ses))
    rows.append(dict(n=n,
                     mean_correct=bf.mean(), sd_correct=bf.std(ddof=1), bias_correct=bf.mean()-B_TRUE,
                     rmse_correct=np.sqrt(((bf-B_TRUE)**2).mean()), cover95_correct=np.mean(np.abs(bf-B_TRUE) <= 1.96*sef),
                     mean_omitted=bs.mean(), sd_omitted=bs.std(ddof=1), bias_omitted=bs.mean()-B_TRUE,
                     rmse_omitted=np.sqrt(((bs-B_TRUE)**2).mean()), cover95_omitted=np.mean(np.abs(bs-B_TRUE) <= 1.96*ses)))
mc = pd.DataFrame(rows)
print(mc.to_string(index=False))
print(f"\nAsymptotic target for the misspecified estimator: {B_TRUE + theory_bias:.4f}")

In [ ]:
# --- 1e(iii) the bias tracks corr(X, Z) exactly ------------------------------------------------
sens = []
for r in [-0.9, -0.5, -0.2, 0.0, 0.2, 0.5, 0.7, 0.9]:
    rng = np.random.default_rng(SEED + int(1000*r) + 7)
    bs = [sm.OLS(*(lambda t: (t[2], sm.add_constant(t[0])))(draw(5000, rng, rho=r))).fit().params[1]
          for _ in range(2000)]
    sens.append(dict(corr_XZ=r, simulated_mean_b=np.mean(bs), predicted_b=B_TRUE + C_TRUE*r,
                     simulated_bias=np.mean(bs)-B_TRUE, predicted_bias=C_TRUE*r))
sens = pd.DataFrame(sens)
print(sens.to_string(index=False))

In [ ]:
# --- 1e figure ---------------------------------------------------------------------------------
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
rng = np.random.default_rng(SEED + 1)
bf_s, bs_s = [], []
for _ in range(3000):
    X, Z, Y = draw(200, rng)
    bf_s.append(sm.OLS(Y, sm.add_constant(np.column_stack([X, Z]))).fit().params[1])
    bs_s.append(sm.OLS(Y, sm.add_constant(X)).fit().params[1])

ax[0].hist(bf_s, 45, alpha=.75, color=BLUE, label="Both predictors included")
ax[0].hist(bs_s, 45, alpha=.75, color=RED,  label="Z omitted")
ax[0].axvline(B_TRUE, color="k", lw=1.6, label=f"True b = {B_TRUE}")
ax[0].axvline(B_TRUE+theory_bias, color=RED, ls="--", lw=1.6, label=f"b + c·ρ = {B_TRUE+theory_bias:.2f}")
ax[0].set(xlabel="Estimate of b", ylabel="Frequency out of 3,000 samples",
          title="(i) Sampling distributions at n = 200")
ax[0].legend(fontsize=7, frameon=False)

ax[1].axhline(B_TRUE, color="k", lw=1.5, label=f"True b = {B_TRUE}")
ax[1].axhline(B_TRUE+theory_bias, color=RED, ls="--", lw=1.5, label=f"Asymptotic limit = {B_TRUE+theory_bias:.2f}")
ax[1].errorbar(mc.n, mc.mean_correct, yerr=1.96*mc.sd_correct, fmt="o-", color=BLUE, ms=4, capsize=2,
               label="Both predictors included")
ax[1].errorbar(mc.n, mc.mean_omitted, yerr=1.96*mc.sd_omitted, fmt="s-", color=RED, ms=4, capsize=2,
               label="Z omitted")
ax[1].set(xscale="log", xlabel="Sample size n (logarithmic scale)", ylabel="Mean estimate of b ± 1.96 SD",
          title="(ii) A larger sample does not remove the bias")
ax[1].legend(fontsize=7, frameon=False)

ax[2].plot(sens.corr_XZ, sens.simulated_mean_b, "o", color=RED, ms=6, label="Simulated mean, Z omitted")
ax[2].plot(sens.corr_XZ, sens.predicted_b, "-", color="k", lw=1.4, label="Theory: b + c·corr(X, Z)")
ax[2].axhline(B_TRUE, color=GREY, ls=":", lw=1.4, label=f"True b = {B_TRUE}")
ax[2].set(xlabel="corr(X, Z)", ylabel="Mean estimate of b",
          title="(iii) Sign and size of the bias follow corr(X, Z)")
ax[2].legend(fontsize=7, frameon=False)
save_fig("fig_1e_omitted_variable_bias")

### Baseline replication — single draw per sample size (Answer.pdf)

The group's first pass at 1e used a different known model, $Y_i = 2 + 1.5X_i + 2Z_i + e_i$, with
$X_i = 0.7Z_i + \sqrt{1-0.7^2}\,u_i$ so that $\operatorname{corr}(X,Z)=0.7$, and one sample at each
size. Theory gives $\operatorname{plim}\hat b_{\text{short}} = 1.5 + 2(0.7) = 2.9$. The cell reproduces
the table in the written answer exactly (legacy seed 610).

In [ ]:
np.random.seed(610)
rows = []
for n in [100, 500, 1000, 5000]:
    Zb = np.random.normal(0, 1, n)
    Xb = 0.7*Zb + np.sqrt(1 - 0.7**2)*np.random.normal(0, 1, n)
    Yb = 2 + 1.5*Xb + 2*Zb + np.random.normal(0, 1, n)
    rows.append(dict(n=n,
                     b_with_X_and_Z=sm.OLS(Yb, sm.add_constant(np.column_stack([Xb, Zb]))).fit().params[1],
                     b_with_X_only=sm.OLS(Yb, sm.add_constant(Xb)).fit().params[1]))
print("True b = 1.5;  probability limit with Z omitted = 1.5 + 2(0.7) = 2.9\n")
print(pd.DataFrame(rows).to_string(index=False))

**Results.** With Z included, the estimate of $b$ is centred on the true value 2 at every sample size (bias below 0.004) and its spread shrinks like $1/\sqrt{n}$. With Z omitted, it is centred on $b + c\rho = 3.05$ at every sample size: the bias of about 1.05 is the same at $n = 25$ and at $n = 20{,}000$. Only the spread shrinks, so the 95% interval covers the true $b$ in 9% of samples at $n = 25$ and in none from $n = 200$ onwards. **A larger sample does not cure omitted-variable bias; it makes the wrong answer more precise.** The bias moves one-for-one with $c\cdot\operatorname{corr}(X,Z)$ and disappears only when $\operatorname{corr}(X,Z) = 0$. The baseline replication shows the same pattern with a different model: $\hat b \approx 2.8$–$2.9$ against a true 1.5, with no improvement as $n$ grows.